## Combine simple classification and NN model

Also move from per-cell to per-patient classification

In [ ]:
import numpy as np  # noqa: I001
from MLstatkit import Bootstrapping
from sklearn.metrics import roc_auc_score

from simple_classn_and_comb_cells import (
    eval_best_nn_model,
    log_regr,
    make_cell_results_df,
    save_cell_results_df,
)

## Choose standard or balanced accuracy metric

In [ ]:
balanced_accuracy = True

if balanced_accuracy:
    from sklearn.metrics import balanced_accuracy_score as accuracy_score
else:
    from sklearn.metrics import accuracy_score

## Logistic regression classification (repeat of per-cell logistic regression results in other notebook)

c=100, l1_ratio= .25, max_iter= 5000 penalty=elasticnet

In [ ]:
test_folds, lr_test_files, lr_pred_probs = log_regr()

## Best NN model

In [ ]:
(
    nn_final_test_file_list,
    nn_final_test_probs,
    nn_final_test_predictions,
    nn_final_test_gt,
) = eval_best_nn_model()

## Combine probs and calculate metrics

In [ ]:
df = make_cell_results_df(
    lr_test_files,
    lr_pred_probs,
    nn_final_test_file_list,
    nn_final_test_probs,
    nn_final_test_gt
)

In [ ]:
simple_rocs = []
NN_rocs = []
comb_rocs = []

simple_accs = []
NN_accs = []
comb_accs = []

for fold in range(5):
    test_fold_files = test_folds[fold]
    test_fold_files = [x.rstrip(".parquet") for x in test_fold_files]
    fold_df = df[df["File"].isin(test_fold_files)]

    simple_roc = roc_auc_score(fold_df["GT"], fold_df["simple_prob"])
    NN_roc = roc_auc_score(fold_df["GT"], fold_df["NN_prob"])
    comb_roc = roc_auc_score(fold_df["GT"], fold_df["Combined_prob"])

    simple_acc = accuracy_score(
        fold_df["GT"], np.where(fold_df["simple_prob"] > 0.5, 1, 0)
    )
    NN_acc = accuracy_score(fold_df["GT"], np.where(fold_df["NN_prob"] > 0.5, 1, 0))
    comb_acc = accuracy_score(
        fold_df["GT"], np.where(fold_df["Combined_prob"] > 0.5, 1, 0)
    )

    simple_rocs.append(simple_roc)
    NN_rocs.append(NN_roc)
    comb_rocs.append(comb_roc)

    simple_accs.append(simple_acc)
    NN_accs.append(NN_acc)
    comb_accs.append(comb_acc)

print("Mean simple ROC")
print(np.mean(simple_rocs))
print(np.std(simple_rocs, ddof=1))
print("------------\n")

print("Mean NN ROC")
print(np.mean(NN_rocs))
print(np.std(NN_rocs, ddof=1))
print("------------\n")

print("Mean combined ROC")
print(np.mean(comb_rocs))
print(np.std(comb_rocs, ddof=1))
print("------------\n")

print("Mean simple acc")
print(np.mean(simple_accs))
print(np.std(simple_accs, ddof=1))
print("------------\n")

print("Mean NN acc")
print(np.mean(NN_accs))
print(np.std(NN_accs, ddof=1))
print("------------\n")

print("Mean combined acc")
print(np.mean(comb_accs))
print(np.std(comb_accs, ddof=1))
print("------------\n")

## Aggregate per patient

In [ ]:
print(df.columns)

agg_df = (
    df[["GT", "simple_prob", "NN_prob", "Combined_prob", "patient"]]
    .groupby("patient")
    .mean()
)
target = agg_df["GT"]

# Simple
simple_prob = agg_df["simple_prob"]
simple_pred = [1 if x > 0.5 else 0 for x in simple_prob]
print("Simple")
print("AUROC: ", roc_auc_score(target, simple_prob))
print("BA : ", accuracy_score(target, simple_pred))

# NN
nn_prob = agg_df["NN_prob"]
nn_pred = [1 if x > 0.5 else 0 for x in nn_prob]
print("NN")
print("AUROC: ", roc_auc_score(target, nn_prob))
print("BA : ", accuracy_score(target, nn_pred))

# Combined
comb_prob = agg_df["Combined_prob"]
comb_pred = [1 if x > 0.5 else 0 for x in comb_prob]
print("Combined")
print("AUROC: ", roc_auc_score(target, comb_prob))
print("BA : ", accuracy_score(target, comb_pred))


In [ ]:
# Loop through multiple metrics
for score in ["roc_auc", "recall"]:
    original_score, conf_lower, conf_upper = Bootstrapping(
        target, simple_prob, score, threshold=0.5, n_bootstraps=1000
    )
    print(
        f"Simple: {score.upper()} original score: {original_score:.3f}, confidence interval: [{conf_lower:.3f} - {conf_upper:.3f}]"
    )

    original_score, conf_lower, conf_upper = Bootstrapping(
        target, nn_prob, score, threshold=0.5, n_bootstraps=1000
    )
    print(
        f"NN: {score.upper()} original score: {original_score:.3f}, confidence interval: [{conf_lower:.3f} - {conf_upper:.3f}]"
    )

    original_score, conf_lower, conf_upper = Bootstrapping(
        target, comb_prob, score, threshold=0.5, n_bootstraps=1000
    )
    print(
        f"Combined: {score.upper()} original score: {original_score:.3f}, confidence interval: [{conf_lower:.3f} - {conf_upper:.3f}]"
    )

In [ ]:
# save df
# output_folder = "../output/combined_classification"
#if not os.path.exists(output_folder):
#    os.makedirs(output_folder)
#df.to_csv(os.path.join(output_folder, "k_fold_df.csv"))

save_cell_results_df(df)